# Traditional IR Approach: TF-IDF Vector Space Model

### CLEF 2025 - CheckThat! Lab  - Task 4 Scientific Web Discourse - Subtask 4b (Scientific Claim Source Retrieval)

This notebook implements a traditional Information Retrieval (IR) approach using the Vector Space Model with TF-IDF weighting, as an alternative to the BM25 baseline.

It includes:
- Code to load and process the dataset
- Implementation of a TF-IDF Vector Space Model
- Code to evaluate the model using MRR@k
- Comparison with the BM25 baseline


# 1) Importing Libraries and Data

In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt

## 1.a) Import the collection set
The collection set contains metadata of CORD-19 academic papers.

In [2]:
# 1) Download the collection set from the Gitlab repository: https://gitlab.com/checkthat_lab/clef2025-checkthat-lab/-/tree/main/task4/subtask_4b
# 2) Drag and drop the downloaded file to the "Files" section (left vertical menu on Colab)
# 3) Modify the path to your local file path
PATH_COLLECTION_DATA = 'subtask4b_collection_data.pkl' #MODIFY PATH
df_collection = pd.read_pickle(PATH_COLLECTION_DATA)

In [3]:
df_collection.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7718 entries, 162 to 1056448
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   cord_uid          7718 non-null   object        
 1   source_x          7718 non-null   object        
 2   title             7718 non-null   object        
 3   doi               7677 non-null   object        
 4   pmcid             4959 non-null   object        
 5   pubmed_id         6233 non-null   object        
 6   license           7718 non-null   object        
 7   abstract          7718 non-null   object        
 8   publish_time      7715 non-null   object        
 9   authors           7674 non-null   object        
 10  journal           6668 non-null   object        
 11  mag_id            0 non-null      float64       
 12  who_covidence_id  528 non-null    object        
 13  arxiv_id          20 non-null     object        
 14  label             7718 n

In [4]:
df_collection.head()

,cord_uid,source_x,title,doi,pmcid,pubmed_id,license,abstract,publish_time,authors,journal,mag_id,who_covidence_id,arxiv_id,label,time,timet
162,umvrwgaw,PMC,Professional and Home-Made Face Masks Reduce E...,10.1371/journal.pone.0002618,PMC2440799,18612429,cc-by,BACKGROUND: Governments are preparing for a po...,2008-07-09,"van der Sande, Marianne; Teunis, Peter; Sabel,...",PLoS One,NaN,NaN,NaN,umvrwgaw,2008-07-09,1215561600
611,spiud6ok,PMC,The Failure of R (0),10.1155/2011/527610,PMC3157160,21860658,cc-by,"The basic reproductive ratio, R (0), is one of...",2011-08-16,"Li, Jing; Blakeley, Daniel; Smith?, Robert J.",Comput Math Methods Med,NaN,NaN,NaN,spiud6ok,2011-08-16,1313452800
918,aclzp3iy,PMC,Pulmonary sequelae in a patient recovered from...,10.4103/0970-2113.99118,PMC3424870,22919170,cc-by-nc-sa,The pandemic of swine flu (H1N1) influenza spr...,2012,"Singh, Virendra; Sharma, Bharat Bhushan; Patel...",Lung India,NaN,NaN,NaN,aclzp3iy,2012-01-01,1325376000
993,ycxyn2a2,PMC,What was the primary mode of smallpox transmis...,10.3389/fcimb.2012.00150,PMC3509329,23226686,cc-by,The mode of infection transmission has profoun...,2012-11-29,"Milton, Donald K.",Front Cell Infect Microbiol,NaN,NaN,NaN,ycxyn2a2,2012-11-29,1354147200
1053,zxe95qy9,PMC,"Lessons from the History of Quarantine, from P...",10.3201/eid1902.120312,PMC3559034,23343512,no-cc,"In the new millennium, the centuries-old strat...",2013-02-03,"Tognotti, Eugenia",Emerg Infect Dis,NaN,NaN,NaN,zxe95qy9,2013-02-03,1359849600


## 1.b) Import the query set

The query set contains tweets with implicit references to academic papers from the collection set.

In [5]:
# 1) Download the query tweets from the Gitlab repository: https://gitlab.com/checkthat_lab/clef2025-checkthat-lab/-/tree/main/task4/subtask_4b?ref_type=heads
# 2) Drag and drop the downloaded file to the "Files" section (left vertical menu on Colab)
# 3) Modify the path to your local file path
PATH_QUERY_TRAIN_DATA = 'subtask4b_query_tweets_train.tsv' #MODIFY PATH
PATH_QUERY_DEV_DATA = 'subtask4b_query_tweets_dev.tsv' #MODIFY PATH

df_query_train = pd.read_csv(PATH_QUERY_TRAIN_DATA, sep = '\t')
df_query_dev = pd.read_csv(PATH_QUERY_DEV_DATA, sep = '\t')

In [6]:
df_query_train.head()

,post_id,tweet_text,cord_uid
0,0,Oral care in rehabilitation medicine: oral vul...,htlvpvz5
1,1,this study isn't receiving sufficient attentio...,4kfl29ul
2,2,"thanks, xi jinping. a reminder that this study...",jtwb17u8
3,3,Taiwan - a population of 23 million has had ju...,0w9k8iy1
4,4,Obtaining a diagnosis of autism in lower incom...,tiqksd69


In [7]:
df_query_dev.head()

,post_id,tweet_text,cord_uid
0,16,covid recovery: this study from the usa reveal...,3qvh482o
1,69,"""Among 139 clients exposed to two symptomatic ...",r58aohnu
2,73,I recall early on reading that researchers who...,sts48u9i
3,93,You know you're credible when NIH website has ...,3sr2exq9
4,96,Resistance to antifungal medications is a grow...,ybwwmyqy


# 2) Building the TF-IDF Vector Space Model

In this section, we'll implement a traditional Vector Space Model, which falls under the category of traditional IR using TF-IDF. This is a classic approach in information retrieval that represents documents and queries as vectors in a high-dimensional space, where each dimension corresponds to a term in the vocabulary.

In [8]:
# Create the corpus for TF-IDF model
print("Creating corpus...")
corpus = df_collection[:][['title', 'abstract']].apply(lambda x: f"{x['title']} {x['abstract']}", axis=1).tolist()
cord_uids = df_collection[:]['cord_uid'].tolist()

# Build the TF-IDF model
print("Building TF-IDF model...")
# Creating a TF-IDF vectorizer with common preprocessing
vectorizer = TfidfVectorizer(
    lowercase=True,  # Convert to lowercase
    stop_words='english',  # Remove English stopwords
    min_df=2,  # Ignore terms that appear in less than 2 documents
    max_df=0.95,  # Ignore terms that appear in more than 95% of documents
    ngram_range=(1, 2),  # Consider both unigrams and bigrams
    norm='l2'  # L2 normalization of vectors (for cosine similarity)
)

# Fit the vectorizer on the corpus
tfidf_matrix = vectorizer.fit_transform(corpus)
print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")

Creating corpus...
Building TF-IDF model...
TF-IDF matrix shape: (7718, 141861)


# 3) Retrieving Documents with the TF-IDF Model

Next, we'll define a function to retrieve the top k most relevant documents for a given query using TF-IDF vectors and cosine similarity.

In [2]:
def get_top_cord_uids_tfidf(query, top_k=5):
    """
    Retrieve top documents for a query using TF-IDF and cosine similarity
    """
    # Transform the query using the same vectorizer
    query_vector = vectorizer.transform([query])
    
    # Calculate cosine similarity between query and all documents
    cosine_similarities = cosine_similarity(query_vector, tfidf_matrix).flatten()
    
    # Get indices of top k most similar documents
    top_indices = cosine_similarities.argsort()[-top_k:][::-1]
    
    # Return the cord_uids of the top documents
    return [cord_uids[i] for i in top_indices]

In [10]:
# Retrieve top documents for each query
print("Retrieving documents for train set...")
df_query_train['tfidf_topk'] = df_query_train['tweet_text'].apply(lambda x: get_top_cord_uids_tfidf(x))

print("Retrieving documents for dev set...")
df_query_dev['tfidf_topk'] = df_query_dev['tweet_text'].apply(lambda x: get_top_cord_uids_tfidf(x))

Retrieving documents for train set...
Retrieving documents for dev set...


# 4) Evaluating the TF-IDF Model

We'll use the Mean Reciprocal Rank (MRR@k) metric to evaluate our TF-IDF model, which is the same evaluation metric used in the original notebook.

In [11]:
def get_performance_mrr(data, col_gold, col_pred, list_k=[1, 5, 10]):
    """
    Calculate MRR@k for the predictions
    """
    d_performance = {}
    for k in list_k:
        data["in_topx"] = data.apply(
            lambda x: (1/([i for i in x[col_pred][:k]].index(x[col_gold]) + 1) 
                      if x[col_gold] in [i for i in x[col_pred][:k]] else 0), 
            axis=1
        )
        d_performance[k] = data["in_topx"].mean()
    return d_performance

In [12]:
# Calculate and print results
print("Evaluating results...")
results_train = get_performance_mrr(df_query_train, 'cord_uid', 'tfidf_topk')
results_dev = get_performance_mrr(df_query_dev, 'cord_uid', 'tfidf_topk')

print(f"TF-IDF results on the train set: {results_train}")
print(f"TF-IDF results on the dev set: {results_dev}")

Evaluating results...
TF-IDF results on the train set: {1: 0.4938146736170544, 5: 0.5557120257268082, 10: 0.5557120257268082}
TF-IDF results on the dev set: {1: 0.49357142857142855, 5: 0.5549761904761905, 10: 0.5549761904761905}


# 5) Comparison with BM25 Baseline

Now we'll implement and run the BM25 baseline to compare with our TF-IDF approach.

In [13]:
# Install the rank_bm25 package if not already installed
!pip install rank_bm25
from rank_bm25 import BM25Okapi

In [ ]:
# Run the BM25 baseline for comparison
print("Running BM25 baseline for comparison...")

tokenized_corpus = [doc.split(' ') for doc in corpus]
bm25 = BM25Okapi(tokenized_corpus)

def get_top_cord_uids_bm25(query, top_k=5):
    tokenized_query = query.split(' ')
    doc_scores = bm25.get_scores(tokenized_query)
    indices = np.argsort(-doc_scores)[:top_k]
    return [cord_uids[x] for x in indices]

df_query_train['bm25_topk'] = df_query_train['tweet_text'].apply(lambda x: get_top_cord_uids_bm25(x))
df_query_dev['bm25_topk'] = df_query_dev['tweet_text'].apply(lambda x: get_top_cord_uids_bm25(x))

bm25_results_train = get_performance_mrr(df_query_train, 'cord_uid', 'bm25_topk')
bm25_results_dev = get_performance_mrr(df_query_dev, 'cord_uid', 'bm25_topk')

print(f"BM25 results on the train set: {bm25_results_train}")
print(f"BM25 results on the dev set: {bm25_results_dev}")

# 6) Visualizing the Results Comparison

Let's create visualizations to compare the performance of TF-IDF and BM25.

In [ ]:
# Create comparison visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Train set comparison
x = list(results_train.keys())
y1 = list(results_train.values())
y2 = list(bm25_results_train.values())

ax1.plot(x, y1, marker='o', label='TF-IDF')
ax1.plot(x, y2, marker='s', label='BM25')
ax1.set_title('MRR@k on Train Set')
ax1.set_xlabel('k')
ax1.set_ylabel('MRR@k')
ax1.set_xticks(x)
ax1.legend()
ax1.grid(True, alpha=0.3)

# Dev set comparison
x = list(results_dev.keys())
y1 = list(results_dev.values())
y2 = list(bm25_results_dev.values())

ax2.plot(x, y1, marker='o', label='TF-IDF')
ax2.plot(x, y2, marker='s', label='BM25')
ax2.set_title('MRR@k on Dev Set')
ax2.set_xlabel('k')
ax2.set_ylabel('MRR@k')
ax2.set_xticks(x)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('comparison.png')
plt.show()

# 7) Exporting Results for Submission

Finally, we'll prepare the results for submission on Codalab.

In [ ]:
# Export TF-IDF results for submission
df_query_dev['preds'] = df_query_dev['tfidf_topk'].apply(lambda x: x[:5])
df_query_dev[['post_id', 'preds']].to_csv('tfidf_predictions.tsv', index=None, sep='\t')
print("TF-IDF results exported to tfidf_predictions.tsv")

# Optional: Export BM25 results as well for comparison
df_query_dev['preds'] = df_query_dev['bm25_topk'].apply(lambda x: x[:5])
df_query_dev[['post_id', 'preds']].to_csv('bm25_predictions.tsv', index=None, sep='\t')
print("BM25 results exported to bm25_predictions.tsv")

# 8) Analysis and Conclusion

In this notebook, we have implemented a traditional Vector Space Model with TF-IDF weighting for the scientific claim source retrieval task. We have also compared its performance with the BM25 baseline.

The TF-IDF model includes several enhancements over simple term matching:
- Stopword removal to filter out common words
- Consideration of both individual terms and bigrams
- Document frequency filtering to focus on informative terms
- Vector normalization for better similarity comparisons

By comparing the MRR@k scores, we can see the relative strengths of each approach. TF-IDF is a classic approach that serves as a strong baseline in many IR tasks, while BM25 often performs better due to its more sophisticated term weighting scheme that considers document length normalization.

In [7]:
import pandas as pd

# Lade die Entwicklungs-Queries (Tweets) über den vollständigen Pfad
df_query_dev = pd.read_csv(r"C:\Users\jakob\OneDrive\001_TU Data Science\11_AIR\subtask4b_query_tweets_dev.tsv", sep='\t')
# Optional: Anzeigen
df_query_dev.head()

import pandas as pd

# Pfad zur .pkl-Datei (falls sie im gleichen Ordner liegt)
collection_df = pd.read_pickle("subtask4b_collection_data.pkl")

# Optional: anschauen
collection_df.head()

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Dokumente (Papers) vorbereiten
docs = collection_df['title'].fillna('') + ' ' + collection_df['abstract'].fillna('')
doc_ids = collection_df['cord_uid'].tolist()

# Queries (Tweets)
queries = df_query_dev['tweet_text'].fillna('')
query_ids = df_query_dev['post_id'].tolist()

# TF-IDF Vektorisierung
vectorizer = TfidfVectorizer(stop_words='english', max_features=10000, ngram_range=(1,2))
doc_tfidf = vectorizer.fit_transform(docs)
query_tfidf = vectorizer.transform(queries)

# Kosinusähnlichkeit berechnen
similarity_matrix = cosine_similarity(query_tfidf, doc_tfidf)

# Top-K Ergebnisse extrahieren
top_k = 5
tfidf_topk = []
for row in similarity_matrix:
    top_indices = np.argsort(row)[::-1][:top_k]
    tfidf_topk.append([doc_ids[i] for i in top_indices])

# Ergebnisse speichern
df_query_dev['tfidf_topk'] = tfidf_topk

In [8]:
# Ergebnisse exportieren für Codalab
df_query_dev['preds'] = df_query_dev['tfidf_topk'].apply(lambda x: x[:5])
df_query_dev[['post_id', 'preds']].to_csv('tfidf_predictions.tsv', index=None, sep='\t')
print("TF-IDF results exported to tfidf_predictions.tsv")

TF-IDF results exported to tfidf_predictions.tsv
